# Brain Tumor Segmentation - Real BraTS Data Training
**Train on REAL BraTS dataset with FREE GPU from Colab!**

## Quick Start:
1. Make sure GPU is enabled: Runtime → Change runtime type → GPU
2. Get Kaggle API key: kaggle.com/settings/account → Create API Token
3. Run CELL 1 first (uploads kaggle.json)
4. Run other cells in order
5. Training takes ~40-50 hours on free T4 GPU

**This notebook downloads the REAL 25-30GB BraTS dataset**

## CELL 1: Upload Kaggle API Credentials

In [ ]:
from google.colab import files
import os

print("="*60)
print("STEP 1: Setup Kaggle API Credentials")
print("="*60 + "\n")

print("Instructions:")
print("1. Go to: https://kaggle.com/settings/account")
print("2. Scroll down and click 'Create New API Token'")
print("3. This downloads 'kaggle.json'")
print("4. Upload it in the dialog that appears below\n")

uploaded = files.upload()

if 'kaggle.json' in uploaded:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("\n✓ Kaggle credentials configured!")
else:
    print("\n❌ kaggle.json not found. Please upload it.")

## CELL 2: Check GPU & Mount Drive

In [ ]:
import torch
from google.colab import drive

print("="*60)
print("STEP 2: Check GPU and Mount Google Drive")
print("="*60 + "\n")

# Check GPU
print("GPU Status:")
print(f"  GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU Model: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU Memory: {props.total_memory / 1e9:.2f} GB")
else:
    print("\n⚠️  NO GPU! Enable it:")
    print("   Runtime → Change runtime type → GPU")

# Mount Drive
print("\n" + "="*60)
print("Mounting Google Drive...")
print("="*60)

drive.mount('/content/drive', force_remount=True)
print("✓ Drive mounted!")

## CELL 3: Check Available Storage

In [ ]:
import os
import subprocess

print("="*60)
print("STEP 3: Check Storage Space")
print("="*60 + "\n")

# Check Colab local storage
result = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
print("Colab Local Storage (/content/):")
print(result.stdout)

print("\nStorage needed for BraTS:")
print("  Download (zip):  ~7 GB")
print("  Extract:         ~25-30 GB")
print("  Total needed:    ~32-37 GB")
print("  You have:        ~200 GB ✓")
print("\n✓ Plenty of space available!")

## CELL 4: Clone Repository & Install Dependencies

In [ ]:
import os

print("="*60)
print("STEP 4: Clone Repository")
print("="*60 + "\n")

# Create working directory
os.makedirs('/content/workspace', exist_ok=True)
os.chdir('/content/workspace')

# Clone repo
if not os.path.exists('Brain-Tumour-Segmentation'):
    print("Cloning repository...")
    !git clone https://github.com/karthikpagnis/Brain-Tumour-Segmentation.git
    print("✓ Repository cloned!")
else:
    print("✓ Repository already exists")

os.chdir('Brain-Tumour-Segmentation')
print(f"\nWorking directory: {os.getcwd()}")

In [ ]:
print("="*60)
print("STEP 5: Install Dependencies")
print("="*60 + "\n")

print("Installing PyTorch (CUDA 11.8)...")
!pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

print("Installing medical imaging libraries...")
!pip install -q 'numpy>=2.0.0' 'nibabel>=5.1.0' scipy scikit-image

print("Installing Kaggle API...")
!pip install -q kaggle

print("Installing training tools...")
!pip install -q 'pytorch-lightning>=2.2.0' tensorboard tqdm

print("Installing other utilities...")
!pip install -q pandas matplotlib scikit-learn

print("\n" + "="*60)
print("Verifying installations...")
print("="*60)

import torch
import numpy as np
print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ NumPy: {np.__version__}")
print(f"✓ GPU: {torch.cuda.is_available()}")
print("\n✓ All dependencies installed!")

## CELL 5: Download & Extract BraTS Dataset (Option B Helper)

In [ ]:
import os
import zipfile
import subprocess

def setup_brats_data(extract_to='/content/data', verbose=True):
    """
    Download & setup BraTS dataset from Kaggle
    
    Args:
        extract_to: Directory to extract data
        verbose: Print progress messages
    
    Returns:
        Path to extracted data
    """
    
    if verbose:
        print("="*60)
        print("STEP 6: Download & Extract BraTS Dataset")
        print("="*60 + "\n")
    
    # Create extraction directory
    os.makedirs(extract_to, exist_ok=True)
    
    # Download
    if verbose:
        print("Downloading BraTS dataset from Kaggle...")
        print("(This takes 5-15 minutes depending on internet speed)\n")
    
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', 'awsaf49/brats2020-training-data', '-q'],
        cwd='/content/workspace/Brain-Tumour-Segmentation'
    )
    
    if result.returncode != 0:
        print("❌ Download failed. Did you upload kaggle.json in CELL 1?")
        return None
    
    if verbose:
        print("✓ Download complete!\n")
        print("Extracting dataset...")
        print("(This takes 5-10 minutes)\n")
    
    # Extract
    zip_path = '/content/workspace/Brain-Tumour-Segmentation/brats2020-training-data.zip'
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_to)
    
    if verbose:
        print("✓ Extraction complete!\n")
        print("Cleaning up (deleting zip file)...")
    
    # Cleanup
    os.remove(zip_path)
    
    data_path = os.path.join(extract_to, 'BRATS20_Training_Data')
    
    if verbose:
        print(f"✓ Cleanup complete!\n")
        print("="*60)
        print("✅ BraTS Dataset Ready!")
        print("="*60)
        print(f"Location: {data_path}")
        print(f"Size: ~25-30 GB")
        print(f"Cases: ~370 training samples")
    
    return data_path

# Run the setup
brats_path = setup_brats_data()

if brats_path:
    print(f"\nData ready for training at: {brats_path}")

## CELL 6: Optimize Config for Colab

In [ ]:
print("="*60)
print("STEP 7: Optimize Configuration for Colab T4")
print("="*60 + "\n")

# Read config
with open('config.py', 'r') as f:
    config = f.read()

# Optimize for free tier Colab
replacements = {
    'BATCH_SIZE = 16': 'BATCH_SIZE = 8',
    'NUM_EPOCHS = 100': 'NUM_EPOCHS = 50',
    'NUM_WORKERS = 4': 'NUM_WORKERS = 2',
    'EARLY_STOPPING_PATIENCE = 20': 'EARLY_STOPPING_PATIENCE = 5',
}

for old, new in replacements.items():
    if old in config:
        config = config.replace(old, new)
        print(f"✓ {old} → {new}")

# Write updated config
with open('config.py', 'w') as f:
    f.write(config)

print("\n" + "="*60)
print("Expected Training Time on Free T4 GPU:")
print("="*60)
print("\n  • 50 epochs")
print("  • Batch size: 8")
print("  • Estimated time: 40-50 hours")
print("  • Calendar time: 2-5 days (across multiple sessions)")
print("\n  ⏱️  Note: Sessions timeout after 12 hours on free tier")
print("  ✓ Checkpoints save automatically - you can resume\n")
print("✓ Config optimized!")

## CELL 7: Start Training 🚀

In [ ]:
import sys
import os

print("="*60)
print("STEP 8: START TRAINING")
print("="*60)

# Ensure Python path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print("\nStarting training with real BraTS data...")
print("Expected time: 40-50 hours on free T4 GPU")
print("Batch size: 8 (optimized for GPU memory)")
print("\n" + "="*60 + "\n")

!python training/train.py \
    --experiment_name colab_brats_real \
    --data_dir /content/data/BRATS20_Training_Data \
    --epochs 50 \
    --batch_size 8 \
    --device cuda \
    --log_dir outputs/logs

print("\n" + "="*60)
print("✅ TRAINING COMPLETED!")
print("="*60)
print("\nModel checkpoint saved to:")
print("  checkpoints/colab_brats_real_best.pth")

## CELL 8: View Training Progress (TensorBoard)

In [ ]:
print("Loading TensorBoard...\n")

%load_ext tensorboard
%tensorboard --logdir outputs/logs/tensorboard

print("\nTensorBoard displays:")
print("  • Training & validation loss curves")
print("  • Dice coefficient")
print("  • IoU (Intersection over Union)")
print("  • F1 Score")
print("  • Learning rate")

## CELL 9: Save Model to Google Drive

In [ ]:
import shutil
import os

print("="*60)
print("STEP 9: Save Results to Google Drive")
print("="*60 + "\n")

drive_dir = '/content/drive/MyDrive'
os.makedirs(drive_dir, exist_ok=True)

# Copy model
if os.path.exists('checkpoints/colab_brats_real_best.pth'):
    dest = f'{drive_dir}/colab_brats_real_best.pth'
    shutil.copy('checkpoints/colab_brats_real_best.pth', dest)
    print(f"✓ Model saved to Drive: colab_brats_real_best.pth")
else:
    print("⚠️  Model not found (training may still be running)")

# Copy summary
if os.path.exists('outputs/colab_brats_real_summary.json'):
    dest = f'{drive_dir}/colab_brats_real_summary.json'
    shutil.copy('outputs/colab_brats_real_summary.json', dest)
    print(f"✓ Summary saved to Drive: colab_brats_real_summary.json")

# Copy log
if os.path.exists('outputs/training.log'):
    dest = f'{drive_dir}/training_log.txt'
    shutil.copy('outputs/training.log', dest)
    print(f"✓ Training log saved to Drive: training_log.txt")

print("\n✓ All files backed up to Google Drive!")

## CELL 10: Test Model Inference

In [ ]:
import torch
import sys
import os

print("="*60)
print("STEP 10: Test Model Inference")
print("="*60 + "\n")

# Add to path
if '/content/workspace/Brain-Tumour-Segmentation' not in sys.path:
    sys.path.insert(0, '/content/workspace/Brain-Tumour-Segmentation')

from models.unet_attention import AttentionUNet3D

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# Load model
print("Loading trained model...")
model = AttentionUNet3D().to(device)
checkpoint = torch.load('checkpoints/colab_brats_real_best.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("✓ Model loaded!\n")

# Test inference
print("Running inference test...")
dummy_input = torch.randn(1, 4, 32, 32, 32).to(device)

with torch.no_grad():
    output = model(dummy_input)

print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print("✓ Inference works!\n")

# Model statistics
num_params = sum(p.numel() for p in model.parameters())
print("Model Statistics:")
print(f"  Parameters: {num_params / 1e6:.1f}M")
print(f"  GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## CELL 11: Training Results Summary

In [ ]:
import json
import os

print("="*60)
print("TRAINING RESULTS SUMMARY")
print("="*60 + "\n")

summary_file = 'outputs/colab_brats_real_summary.json'

if os.path.exists(summary_file):
    with open(summary_file, 'r') as f:
        summary = json.load(f)
    
    print(f"Epochs Trained:       {summary.get('epoch', 'N/A')}")
    
    if 'train_loss' in summary and summary['train_loss']:
        print(f"Final Train Loss:     {summary['train_loss'][-1]:.4f}")
    
    if 'val_loss' in summary and summary['val_loss']:
        print(f"Final Val Loss:       {summary['val_loss'][-1]:.4f}")
    
    if 'best_dice' in summary:
        print(f"Best Val Dice:        {summary['best_dice']:.4f}")
    
    if 'best_iou' in summary:
        print(f"Best Val IoU:         {summary['best_iou']:.4f}")
    
    if 'best_f1' in summary:
        print(f"Best Val F1:          {summary['best_f1']:.4f}")
    
    print("\n" + "="*60)
    print("✅ TRAINING COMPLETE!")
    print("="*60)
    print("\nModel is saved in Google Drive:")
    print("  • colab_brats_real_best.pth")
    print("\nNext steps:")
    print("  1. Download model from Drive")
    print("  2. Run inference: python api/main.py")
    print("  3. Try web UI: cd ui && npm install && npm start")
else:
    print("⚠️  Summary file not found.")
    print("Training may still be in progress.")

## CELL 12: Download Results to Local Machine

In [ ]:
from google.colab import files
import os

print("="*60)
print("DOWNLOAD RESULTS TO LOCAL MACHINE")
print("="*60 + "\n")

# Files to download
download_files = [
    'checkpoints/colab_brats_real_best.pth',
    'outputs/colab_brats_real_summary.json',
    'outputs/training.log'
]

print("Preparing files...\n")

for file in download_files:
    if os.path.exists(file):
        print(f"Downloading: {file}")
        files.download(file)
    else:
        print(f"⚠️  {file} not found")

print("\n✓ Check your Downloads folder!")

## ℹ️ Notes & Troubleshooting

### How to Use:
1. Make sure GPU is enabled: **Runtime → Change runtime type → GPU**
2. Run **CELL 1** first to upload kaggle.json
3. Run cells in order (top to bottom)
4. Training takes ~40-50 hours on free T4

### Important:
- **CELL 1 is required**: Upload your Kaggle API credentials
- Checkpoints auto-save every epoch
- If session times out (12 hours), just rerun the training cell
- Model automatically resumes from last checkpoint

### Troubleshooting:

**"No GPU available"**
- Go to: Runtime → Change runtime type → Hardware accelerator: GPU

**"Kaggle credentials not found"**
- Go to: kaggle.com/settings/account
- Click "Create New API Token"
- Upload kaggle.json in CELL 1

**"Out of memory"**
- Reduce BATCH_SIZE in config.py to 4
- Or restart Colab and clear cache

**"Session timed out"**
- This is normal on free tier (12-hour limit)
- Just rerun CELL 7 (training resumes automatically)

### Storage:
- **Colab local (/content/)**: ~200 GB available ✓
- **BraTS dataset**: 25-30 GB
- **Training checkpoints**: ~500 MB each
- **Google Drive**: Save models here for backup